# Turntable: minimal interface demo

This notebook walks through the `Residuals` / `Segment` / `Wheel` contract using `EchoSegment`, a no-op segment that just prints what the Wheel hands it. No real waveform model, no MCMC — just enough to see the plumbing work.

## 1. Build the observed `Residuals`

One frozen object holds both the TDI arrays and the run settings everyone in this run agrees on.

In [ ]:
import numpy as np

from turntable import Residuals, Wheel
from turntable.testing import EchoSegment

rng = np.random.default_rng(0)
n_samples = 1024
channels = ("A", "E", "T")

observed = Residuals(
    tdi={ch: rng.standard_normal(n_samples) for ch in channels},
    sample_rate=0.1,
    n_samples=n_samples,
    channels=channels,
    tdi_generation="2.0",
    observable="fractional_frequency",
    epoch=0.0,
)
observed

## 2. Long names and short shadows

Every derived quantity has two spellings. Use whichever reads better.

In [ ]:
print(f"observation_time = {observed.observation_time} s     (Tobs = {observed.Tobs})")
print(f"sample_rate      = {observed.sample_rate} Hz   (fs   = {observed.fs})")
print(f"sample_interval  = {observed.sample_interval} s     (dt   = {observed.dt})")
print(f"n_samples        = {observed.n_samples}        (N    = {observed.N})")
print(f"freq_resolution  = {observed.frequency_resolution} Hz   (df   = {observed.df})")
print(f"nyquist          = {observed.nyquist_frequency} Hz   (fny  = {observed.fny})")
print(f"epoch            = {observed.epoch} s     (t0   = {observed.t0})")

In [ ]:
Residuals.aliases()

## 3. Typo catcher

Common misspellings point at the canonical spelling instead of failing silently.

In [ ]:
try:
    _ = observed.T_obs
except AttributeError as e:
    print(e)

The Wheel passes one residual around the ring: it hands each segment the current residual and takes back the segment's new one. Segments own all of their internal state, so hold on to the objects you register if you want to read anything back afterwards.

In [ ]:
ucb = EchoSegment(name="ucb")
mbhb = EchoSegment(name="mbhb")

wheel = Wheel(observed)
wheel.add(ucb)
wheel.add(mbhb)

wheel.run(n_iterations=3)

## 5. Segment state stays with the segments

The Wheel holds only the running residual — the observed data minus every segment's current model. Anything else — parameters, chains, step counters — lives on the segment objects themselves; ask them directly. `wheel.residual()` gives the current full residual.

In [ ]:
print(f"ucb steps: {ucb.steps}, mbhb steps: {mbhb.steps}")

full = wheel.residual()  # observed minus every segment's model
print(f"full residual RMS on 'A': {np.sqrt(np.mean(full.tdi['A']**2)):.4f}")

## 6. Attaching the constellation ephemeris

Real data comes with the spacecraft positions it was produced with. That ephemeris rides on `Residuals.orbit` so every segment builds its response from the *same* constellation — and `Residuals` checks at construction that the ephemeris actually spans the observation, catching epoch mismatches (GPS vs zero-based times) before any sampling starts.

`NumericOrbit` needs the `numeric-orbits` extra (`uv sync --extra numeric-orbits`). Here we tabulate a synthetic circular constellation; for real data use `NumericOrbit.from_hdf5` (LDC/Mojito files) or `NumericOrbit.from_lisaorbits`.

In [ ]:
from dataclasses import replace

from turntable import NumericOrbit

# a synthetic 40-day ephemeris: three spacecraft on a 1 AU circle
AU = 1.495978707e11
t_grid = np.linspace(0.0, 40 * 86400.0, 200)
pos = np.zeros((3, t_grid.size, 3))  # (spacecraft, time, xyz), ecliptic metres
for sc in range(3):
    # spacecraft spaced ~0.0167 rad apart on the circle -> ~2.5e9 m arms
    ang = 2 * np.pi * t_grid / (365.25 * 86400.0) + sc * 0.0167
    pos[sc, :, 0] = AU * np.cos(ang)
    pos[sc, :, 1] = AU * np.sin(ang)

orbit = NumericOrbit(t_grid, pos)
print(f"tabulated span: {orbit.t_range}, L = {orbit.L:.3e} m, fstar = {orbit.fstar:.4f} Hz")

observed_with_orbit = replace(observed, orbit=orbit)  # spans the data: fine
print("orbit attached; segments read residual.orbit instead of building their own")

In [ ]:
# the consistency checks in action: a data span the ephemeris does not
# cover is rejected at construction, and out-of-span position queries refuse
# to extrapolate
try:
    replace(observed, orbit=orbit, epoch=39.9 * 86400.0)
except ValueError as e:
    print(f"span check: {e}\n")

try:
    orbit.positions(np.array([100 * 86400.0]))
except ValueError as e:
    print(f"extrapolation check: {e}")

## 7. Where next

- **A real (toy) fit** — [`examples/toy_fit.py`](toy_fit.py) runs two conjugate-Gibbs source segments plus a sampled white-noise segment to convergence: the add-back, the noise contract (`residual.noise_psd`), posterior chains kept on the segment objects, and progress via `Wheel.run(..., on_sweep=...)`.
- **Your own segment** — implement the two-method `start`/`step` protocol (`src/turntable/segment.py` docstrings are the contract), then run `turntable.testing.check_segment(your_segment, toy_observed)` before plugging into a shared campaign.